In [35]:
import json
import sqlite3
import pandas as pd


conn = sqlite3.connect("library")


print("ANSWERS TO BUSINESS QUESTIONS")
print("=" * 60)

q1 = """
SELECT
    m.member_id,
    m.first_name,
    COUNT(c.checkout_id) AS total_checkouts
FROM members AS m
LEFT JOIN checkouts AS c
    ON m.member_id = c.member_id
GROUP BY
    m.member_id,
    m.first_name
ORDER BY
    m.member_id;
"""

q1_result = pd.read_sql_query(q1, conn)

print("\n=== Q1: Member Checkout Counts ===")
print(q1_result)


q2 = """
SELECT
    book_id,
    title,
    author
FROM books
WHERE author LIKE '%Samir%';
"""

q2_result = pd.read_sql_query(q2, conn)

print("\n=== Q2: Books Matching Author Pattern 'Samir' ===")
print(q2_result)


q3 = """
SELECT
    b.book_id,
    b.title,
    b.author,
    COUNT(c.checkout_id) AS checkout_count
FROM checkouts AS c
JOIN books AS b
    ON c.book_id = b.book_id
GROUP BY
    b.book_id,
    b.title,
    b.author
ORDER BY
    checkout_count DESC
LIMIT 5;
"""

q3_result = pd.read_sql_query(q3, conn)

print("\n=== Q3: Five Most Popular Books ===")
print(q3_result)


q4 = """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    COUNT(c.checkout_id) AS checkout_count
FROM checkouts AS c
JOIN members AS m
    ON c.member_id = m.member_id
GROUP BY
    m.member_id,
    m.first_name,
    m.last_name
ORDER BY
    checkout_count DESC
LIMIT 10;
"""

q4_result = pd.read_sql_query(q4, conn)

print("\n=== Q4: Ten Most Active Readers ===")
print(q4_result)


q5 = """
SELECT
    c.member_id,
    c.checkout_date,
    m.neighborhood
FROM checkouts AS c
JOIN members AS m
    ON c.member_id = m.member_id
WHERE m.neighborhood = 'Maadi'
ORDER BY
    c.checkout_date DESC
LIMIT -1 OFFSET 10;
"""

q5_result = pd.read_sql_query(q5, conn)

print("\n=== Q5: Older Maadi Checkout Activity ===")
print(q5_result)


checkouts_df = pd.read_sql_query(
    "SELECT * FROM checkouts",
    conn
)

members_df = pd.read_sql_query(
    "SELECT * FROM members",
    conn
)

books_df = pd.read_sql_query(
    "SELECT * FROM books",
    conn
)

conn.close()


print("\n" + "=" * 60)
print("DATABASE TABLES LOADED")
print("=" * 60)

print("Members:", members_df.shape)
print("Books:", books_df.shape)
print("Checkouts:", checkouts_df.shape)


with open("jeson", "r", encoding="utf-8") as file:
    json_data = json.load(file)

catalog_df = pd.DataFrame(json_data)

print("\nBook Catalog:")
print(catalog_df.shape)



print("\n" + "=" * 60)
print("STAGE 1: MEMBERS + CHECKOUTS")
print("=" * 60)


member_borrow_counts = (
    checkouts_df
    .groupby("member_id")
    .size()
    .reset_index(name="total_checkouts")
)


all_members_borrowing = members_df.merge(
    member_borrow_counts,
    on="member_id",
    how="left"
)

all_members_borrowing["total_checkouts"] = (
    all_members_borrowing["total_checkouts"]
    .fillna(0)
    .astype(int)
)


stage1 = checkouts_df.merge(
    all_members_borrowing,
    on="member_id",
    how="left",
    suffixes=("", "_member")
)


print("Stage 1 shape:", stage1.shape)
print("\nStage 1 sample:")
print(stage1.head())

assert len(stage1) == len(checkouts_df)

print("\nStage 1 check passed:")
print("Original checkouts:", len(checkouts_df))
print("Stage 1 checkouts:", len(stage1))


print("\n" + "=" * 60)
print("STAGE 2: ADD BOOK DETAILS")
print("=" * 60)


full_books = books_df.merge(
    catalog_df,
    on="book_id",
    how="left",
    suffixes=("", "_catalog")
)

stage2 = stage1.merge(
    full_books,
    on="book_id",
    how="left",
    suffixes=("", "_book")
)

stage2["source"] = "Database"


print("Stage 2 shape:", stage2.shape)
print("\nStage 2 sample:")
print(stage2.head())

assert len(stage2) == len(stage1)

print("\nStage 2 check passed:")
print("Stage 1 rows:", len(stage1))
print("Stage 2 rows:", len(stage2))

print("\n" + "=" * 60)
print("STAGE 3: READING KICKOFF")
print("=" * 60)


html_tables = pd.read_html("html")

kickoff_df = html_tables[0].copy()


kickoff_df.columns = [
    "member_id",
    "book_id",
    "checkout_date"
]


kickoff_df["member_id"] = pd.to_numeric(
    kickoff_df["member_id"],
    errors="coerce"
)

kickoff_df["book_id"] = pd.to_numeric(
    kickoff_df["book_id"],
    errors="coerce"
)

kickoff_df["return_date"] = pd.NA

kickoff_df["source"] = "Reading Kickoff"


print("Reading Kickoff rows:", len(kickoff_df))
print("\nReading Kickoff sample:")
print(kickoff_df.head())


kickoff_with_members = kickoff_df.merge(
    all_members_borrowing,
    on="member_id",
    how="left",
    suffixes=("", "_member")
)


kickoff_full = kickoff_with_members.merge(
    full_books,
    on="book_id",
    how="left",
    suffixes=("", "_book")
)


print("\nReading Kickoff combined shape:")
print(kickoff_full.shape)


task1_combined_data = pd.concat(
    [
        stage2,
        kickoff_full
    ],
    ignore_index=True
)


print("\n" + "=" * 60)
print("FINAL DATASET CHECK")
print("=" * 60)


print("Final shape:", task1_combined_data.shape)

print("\nColumns:")
print(task1_combined_data.columns.tolist())

print("\nFirst 5 rows:")
print(task1_combined_data.head())


print("\nDatabase records:", len(stage2))
print("Reading Kickoff records:", len(kickoff_full))
print("Final records:", len(task1_combined_data))

expected_rows = len(stage2) + len(kickoff_full)

assert len(task1_combined_data) == expected_rows

print("\nFinal row count check PASSED.")


print("\nSource counts:")

print(
    task1_combined_data["source"].value_counts()
)


duplicates = task1_combined_data.duplicated().sum()

print("\nDuplicate rows:", duplicates)

output_file = "task1_combined_data.csv"

task1_combined_data.to_csv(
    output_file,
    index=False
)


print("\n" + "=" * 60)
print("TASK 1 COMPLETE")
print("=" * 60)

print(
    f"Combined dataset saved as: {output_file}"
)

ANSWERS TO BUSINESS QUESTIONS

=== Q1: Member Checkout Counts ===
    member_id first_name  total_checkouts
0        1001      Salma                1
1        1002      Fares                2
2        1003     Bassel                9
3        1004      Fares                0
4        1005    Youssef                3
..        ...        ...              ...
75       1076       Dina                7
76       1077       Lina                6
77       1078     Habiba                0
78       1079       Rana               10
79       1080     Bassel                2

[80 rows x 3 columns]

=== Q2: Books Matching Author Pattern 'Samir' ===
   book_id                  title       author
0      531  Voices in the Library  Samir Zohdy
1      532      The Last Bookmark  Samir Zohdy

=== Q3: Five Most Popular Books ===
   book_id                   title          author  checkout_count
0      501         The Silver Kite   Amina Darwish              57
1      507   Fossils and Fireflies     Dalia

In [36]:
df_cleaned = pd.read_csv("task1_combined_data.csv")

print("Original Dataset Shape:")
print(df_cleaned.shape)

print("\nMissing Values Before Cleaning:")
print(df_cleaned.isnull().sum())


print("\n" + "=" * 60)
print("PROBLEM 1: MISSING VALUES")
print("=" * 60)


missing_before = df_cleaned.isnull().sum()

print("\nMissing values before cleaning:")

print(
    missing_before[missing_before > 0]
)


missing_checkout_ids = (
    df_cleaned["checkout_id"].isnull().sum()
)

print(
    "\nMissing checkout_id:",
    missing_checkout_ids
)


if missing_checkout_ids > 0:

    existing_ids = pd.to_numeric(
        df_cleaned["checkout_id"],
        errors="coerce"
    ).dropna()

    if len(existing_ids) > 0:

        next_id = int(existing_ids.max()) + 1

    else:

        next_id = 1

    new_ids = range(
        next_id,
        next_id + missing_checkout_ids
    )

    df_cleaned.loc[
        df_cleaned["checkout_id"].isnull(),
        "checkout_id"
    ] = list(new_ids)



df_cleaned["return_date"] = (
    df_cleaned["return_date"]
    .fillna("Not Provided")
)


df_cleaned["first_name"] = (
    df_cleaned["first_name"]
    .fillna("Unknown")
)

df_cleaned["last_name"] = (
    df_cleaned["last_name"]
    .fillna("Unknown")
)

df_cleaned["neighborhood"] = (
    df_cleaned["neighborhood"]
    .fillna("Unknown")
)

df_cleaned["membership_status"] = (
    df_cleaned["membership_status"]
    .fillna("Unknown")
)

df_cleaned["join_date"] = (
    df_cleaned["join_date"]
    .fillna("Unknown")
)


grade_median = df_cleaned["grade"].median()

df_cleaned["grade"] = (
    df_cleaned["grade"]
    .fillna(grade_median)
)


publication_year_median = (
    df_cleaned["publication_year"].median()
)

df_cleaned["publication_year"] = (
    df_cleaned["publication_year"]
    .fillna(publication_year_median)
)


df_cleaned["total_checkouts"] = (
    df_cleaned["total_checkouts"]
    .fillna(0)
)


print("\n" + "=" * 60)
print("PROBLEM 2: DUPLICATES")
print("=" * 60)


duplicate_count = (
    df_cleaned.duplicated().sum()
)

print(
    "True duplicate rows found:",
    duplicate_count
)


df_cleaned = (
    df_cleaned
    .drop_duplicates()
    .copy()
)


print(
    "Duplicate rows after cleaning:",
    df_cleaned.duplicated().sum()
)


print("\n" + "=" * 60)
print("PROBLEM 3: TEXT INCONSISTENCIES")
print("=" * 60)


print("\nNeighborhood values BEFORE:")

print(
    df_cleaned["neighborhood"]
    .value_counts()
)



df_cleaned["neighborhood"] = (
    df_cleaned["neighborhood"]
    .astype(str)
    .str.strip()
    .str.title()
)


print("\nNeighborhood values AFTER:")

print(
    df_cleaned["neighborhood"]
    .value_counts()
)


print("\nMembership Status BEFORE:")

print(
    df_cleaned["membership_status"]
    .value_counts()
)


df_cleaned["membership_status"] = (
    df_cleaned["membership_status"]
    .astype(str)
    .str.strip()
    .str.capitalize()
)


print("\nMembership Status AFTER:")

print(
    df_cleaned["membership_status"]
    .value_counts()
)

print("\n" + "=" * 60)
print("PROBLEM 4: INVALID MEMBER IDs")
print("=" * 60)


invalid_member_mask = (
    df_cleaned["first_name"].eq("Unknown")
    |
    df_cleaned["last_name"].eq("Unknown")
)


invalid_member_count = (
    invalid_member_mask.sum()
)


print(
    "Records with unmatched member information:",
    invalid_member_count
)


print("\nMember IDs affected:")

print(
    df_cleaned.loc[
        invalid_member_mask,
        "member_id"
    ].unique()
)


df_cleaned = df_cleaned[
    ~invalid_member_mask
].copy()

print("\n" + "=" * 60)
print("FIXING DATA TYPES")
print("=" * 60)


df_cleaned["checkout_id"] = (
    pd.to_numeric(
        df_cleaned["checkout_id"],
        errors="coerce"
    )
    .astype("Int64")
)


df_cleaned["member_id"] = (
    pd.to_numeric(
        df_cleaned["member_id"],
        errors="coerce"
    )
    .astype("Int64")
)


df_cleaned["book_id"] = (
    pd.to_numeric(
        df_cleaned["book_id"],
        errors="coerce"
    )
    .astype("Int64")
)


df_cleaned["grade"] = (
    pd.to_numeric(
        df_cleaned["grade"],
        errors="coerce"
    )
    .round()
    .astype("Int64")
)


df_cleaned["publication_year"] = (
    pd.to_numeric(
        df_cleaned["publication_year"],
        errors="coerce"
    )
    .round()
    .astype("Int64")
)



print("\n" + "=" * 60)
print("FINAL QUALITY CHECK")
print("=" * 60)


print("\nMissing values after cleaning:")

print(
    df_cleaned.isnull().sum()
)


total_missing = (
    df_cleaned.isnull().sum().sum()
)


print(
    "\nTotal missing values:",
    total_missing
)


duplicate_after = (
    df_cleaned.duplicated().sum()
)


print(
    "Duplicate rows after cleaning:",
    duplicate_after
)

output_file = "task2_cleaned_data.csv"


df_cleaned.to_csv(
    output_file,
    index=False
)


print("\n" + "=" * 60)
print("TASK 2 COMPLETE")
print("=" * 60)


print(
    "Saved:",
    output_file
)

print(
    "\nFinal Dataset Shape:",
    df_cleaned.shape
)

print("First 5 rows:")

print(
    df_cleaned.head()
)

if total_missing == 0:
    print(" No missing values remain.")

if duplicate_after == 0:
    print(" No duplicate rows remain.")

print(
    " Cleaned dataset saved successfully."
)

Original Dataset Shape:
(417, 19)

Missing Values Before Cleaning:
checkout_id          26
member_id             0
book_id               0
checkout_date         0
return_date          91
first_name            5
last_name             5
grade                41
neighborhood          5
membership_status     5
join_date            11
total_checkouts       5
title                 0
author                0
genre                 0
pages                 0
publication_year     35
publisher             0
source                0
dtype: int64

PROBLEM 1: MISSING VALUES

Missing values before cleaning:
checkout_id          26
return_date          91
first_name            5
last_name             5
grade                41
neighborhood          5
membership_status     5
join_date            11
total_checkouts       5
publication_year     35
dtype: int64

Missing checkout_id: 26

PROBLEM 2: DUPLICATES
True duplicate rows found: 8
Duplicate rows after cleaning: 0

PROBLEM 3: TEXT INCONSISTENCIES

Neighbo

In [37]:
df_cleaned = pd.read_csv("task2_cleaned_data.csv")

neighborhood_members = (
    df_cleaned
    .groupby("neighborhood")["member_id"]
    .nunique()
    .reset_index(name="member_count")
)

neighborhood_checkouts = (
    df_cleaned
    .groupby("neighborhood")
    .size()
    .reset_index(name="checkout_count")
)


fairness_df = pd.merge(
    neighborhood_members,
    neighborhood_checkouts,
    on="neighborhood"
)

fairness_df["member_pct"] = (
    fairness_df["member_count"]
    / fairness_df["member_count"].sum()
) * 100


fairness_df["checkout_pct"] = (
    fairness_df["checkout_count"]
    / fairness_df["checkout_count"].sum()
) * 100

fairness_df["avg_checkouts_per_member"] = (
    fairness_df["checkout_count"]
    / fairness_df["member_count"]
)

fairness_df["representation_gap"] = (
    fairness_df["checkout_pct"]
    - fairness_df["member_pct"]
)

print("\n" + "=" * 70)
print("NEIGHBORHOOD FAIRNESS SUMMARY")
print("=" * 70)

print(
    fairness_df.to_string(index=False)
)


most_underrepresented = fairness_df.loc[
    fairness_df["representation_gap"].idxmin()
]


print("\n" + "=" * 70)
print("FAIRNESS CHECK")
print("=" * 70)


print(
    "\nNeighborhood with the largest negative representation gap:"
)

print(
    most_underrepresented["neighborhood"]
)


print(
    "\nRepresentation gap:",
    round(
        most_underrepresented["representation_gap"],
        2
    ),
    "percentage points"
)


threshold = -5


if (
    most_underrepresented["representation_gap"]
    < threshold
):

    fairness_finding = (
        f"{most_underrepresented['neighborhood']} "
        "is under-represented because its checkout share "
        "is more than 5 percentage points below its share "
        "of members."
    )

else:

    fairness_finding = (
        "No neighborhood is clearly under-represented. "
        "The checkout share of each neighborhood is reasonably "
        "close to its share of members."
    )


print("\nFairness Finding:")
print(fairness_finding)


fairness_df.to_csv(
    "neighborhood_fairness_summary.csv",
    index=False
)


print(
    "\nSaved: neighborhood_fairness_summary.csv"
)


NEIGHBORHOOD FAIRNESS SUMMARY
neighborhood  member_count  checkout_count  member_pct  checkout_pct  avg_checkouts_per_member  representation_gap
  Heliopolis            13              87   20.000000     21.534653                  6.692308            1.534653
       Maadi            20             114   30.769231     28.217822                  5.700000           -2.551409
   Nasr City            16             101   24.615385     25.000000                  6.312500            0.384615
      Shubra             5              34    7.692308      8.415842                  6.800000            0.723534
     Zamalek            11              68   16.923077     16.831683                  6.181818           -0.091394

FAIRNESS CHECK

Neighborhood with the largest negative representation gap:
Maadi

Representation gap: -2.55 percentage points

Fairness Finding:
No neighborhood is clearly under-represented. The checkout share of each neighborhood is reasonably close to its share of members.

S